# Security Foundations and Tool Policy

**Level:** Beginner · **Duration:** 180–240 min · **Prerequisites:** Basic Python, dicts, dataclasses

## Objectives

1. Identify assets, actors, trust boundaries, and blast radius in an agent system.
2. Observe why caller-asserted identity and unverified approval fail.
3. Build a deterministic pre-execution policy with seven ordered controls.
4. Bind approval to exact arguments, run, policy version, expiry, and an authorized approver.
5. Register a real OpenAI Agents SDK function tool with strict Pydantic input.
6. Produce structured, redacted audit evidence and denominator-aware metrics.
7. Evaluate adversarial, replay, concurrency, and boundary cases.

## Safety Boundary

This lab is a **laptop teaching simulation**.  All data is synthetic.  No
credentials, secrets, API keys, or live services are required.  The side-effect
boundary is an in-memory stub — no real tool calls are made.

## Scenario

An expense assistant can read receipts, calculate totals, preview a claim,
and submit a reimbursement.  The receipt is untrusted input, the employee
identity is authenticated context, and submission is a consequential side
effect requiring human approval.

## Setup

We import the reusable lab module.  The module lives alongside this notebook
in the same directory, so we add the notebook's directory to `sys.path` and
use `importlib` to handle the leading-digit filename.

In [ ]:
import sys, importlib
from pathlib import Path
from datetime import datetime, timedelta, timezone

# Robust import regardless of Jupyter launch directory
for p in [Path("."), Path("curriculum/beginner/01-tool-policy")]:
    if (p / "01_tool_policy.py").exists():
        sys.path.insert(0, str(p.resolve()))
        break

lab = importlib.import_module("01_tool_policy")

# Convenience aliases
ActorContext    = lab.ActorContext
ActionProposal  = lab.ActionProposal
ApprovalReceipt = lab.ApprovalReceipt
ApprovalStore   = lab.ApprovalStore
PolicyEngine    = lab.PolicyEngine
Decision        = lab.Decision
RiskLevel       = lab.RiskLevel
make_actor      = lab.make_actor
proposal_digest = lab.proposal_digest
calculate_evaluation_metrics = lab.calculate_evaluation_metrics

print("Lab module loaded successfully.")

## Part 1 — The Naive Baseline

Before building the full policy, let's see what happens when identity
and approval are **caller-asserted** — the exact mistake the review
identified.

### Why does this fail?

A flat `Request(subject="...", tenant="...", approved=True)` lets
any caller set their own identity, tenant, and approval status.  The
model or an attacker can assert anything.

In [ ]:
# NAIVE BASELINE — do NOT use this in production.
# This demonstrates the exact vulnerability the review identified.

from dataclasses import dataclass

@dataclass(frozen=True)
class NaiveRequest:
    """All fields are caller-supplied — no trusted context."""
    subject: str
    tenant: str
    operation: str
    resource: str
    approved: bool = False

def naive_authorize(req: NaiveRequest) -> str:
    if req.operation not in {"read", "preview", "write"}:
        return "deny: not allowlisted"
    if req.tenant != "tenant-a":
        return "deny: tenant boundary"
    if req.operation == "write" and not req.approved:
        return "pause: approval required"
    return "allow"

# Attack 1: Forged approval — any caller can set approved=True
forged = NaiveRequest("", "tenant-a", "write", "anything", approved=True)
print(f"Forged approval:       {naive_authorize(forged)}")

# Attack 2: Cross-tenant resource — tenant check ignores resource ownership
cross = NaiveRequest("attacker", "tenant-a", "write", "tenant-b/payroll", approved=True)
print(f"Cross-tenant resource: {naive_authorize(cross)}")

print("\n⚠️  Both attacks returned 'allow' — this is the problem.")

### Observations

Both attacks succeeded because:

1. **`approved=True` is caller-asserted** — there is no receipt binding,
   no approver identity, no expiry, and no resource match.
2. **`subject` is never checked** — an empty string or unknown principal
   is treated the same as a known employee.
3. **Resource ownership is not looked up** — the tenant check compares
   the request's tenant field but never verifies that the resource
   actually belongs to that tenant.

The rest of this lab fixes these failures.

## Part 2 — Trusted vs. Untrusted Data Boundaries

The improved design separates data into trust boundaries:

| Structure | Boundary | Authority |
|---|---|---|
| `ActorContext` | Trusted application context | Simulated IAM registry |
| `ActionProposal` | Untrusted | Model output |
| `ResourceMeta` | Trusted lookup | Resource registry |
| `ApprovalReceipt` | Untrusted until verified | Simulated approval service |

**Data type ≠ trust. Provenance and verification establish trust.**

Let's create our actors and proposals.

In [ ]:
NOW = datetime(2026, 1, 15, 12, 0, 0, tzinfo=timezone.utc)

# TRUSTED: Actor contexts created from the authoritative identity registry
acme_employee = make_actor("emp-42", run_id="notebook-run-001")

# An attacker trying to assert their own scopes
unknown_actor = ActorContext(
    subject="hacker-99", tenant="acme",
    scopes=frozenset({"expense:read", "expense:submit"}),
    run_id="notebook-run-002",
)

# UNTRUSTED: Proposals from the model
read_receipt = ActionProposal("read_receipt", "receipt-101")
submit_claim = ActionProposal("submit_claim", "claim-501", {"amount": 250.0})

print(f"Actor: {acme_employee.subject} @ {acme_employee.tenant}")
print(f"Scopes: {sorted(acme_employee.scopes)}")
print(f"Proposal: {read_receipt.operation} on {read_receipt.resource_id}")

## Part 3 — Exercising the Policy Engine

The `PolicyEngine` applies seven checks in order.  Let's evaluate
several scenarios and inspect the structured decisions.

In [ ]:
store = ApprovalStore()
engine = PolicyEngine(budget=20, approval_store=store)

# Scenario 1: Permitted read
d1 = engine.evaluate(acme_employee, read_receipt, now=NOW)
print(f"1. Read receipt:  {d1.state.value:5s}  reason={d1.reason}")

# Scenario 2: Unknown subject
d2 = engine.evaluate(unknown_actor, read_receipt, now=NOW)
print(f"2. Unknown actor: {d2.state.value:5s}  reason={d2.reason}")

# Scenario 3: Cross-tenant resource
cross_tenant = ActionProposal("read_receipt", "receipt-200")
d3 = engine.evaluate(acme_employee, cross_tenant, now=NOW)
print(f"3. Cross-tenant:  {d3.state.value:5s}  reason={d3.reason}")

# Scenario 4: Submit without approval → pause
d4 = engine.evaluate(acme_employee, submit_claim, now=NOW)
print(f"4. No approval:   {d4.state.value:5s}  reason={d4.reason}")

# Scenario 5: Submit with valid approval → allow
valid_receipt = store.issue(
    acme_employee, submit_claim, now=NOW, minutes_valid=30,
)
d5 = engine.evaluate(acme_employee, submit_claim, valid_receipt, now=NOW)
print(f"5. With approval: {d5.state.value:5s}  reason={d5.reason}")

### Interpretation

- **Scenario 1** passes all seven checks → `allow` + execution.
- **Scenario 2** fails at Step 1 (unknown subject) → `deny`, no execution.
- **Scenario 3** fails at Step 3 (resource belongs to Globex, not Acme) → `deny`.
- **Scenario 4** fails at Step 5 (high-risk write, no approval) → `pause`.
- **Scenario 5** passes all checks including approval binding → `allow` + execution.

## Part 4 — Audit Evidence

Every evaluation emits a structured `AuditEvent`.  Let's inspect them.

In [ ]:
print(f"Total audit events: {len(engine.audit_log)}")
print(f"Executed events:    {sum(1 for e in engine.audit_log if e.terminal_state == 'executed')}")
print(f"Blocked events:     {sum(1 for e in engine.audit_log if e.terminal_state == 'blocked')}")
print()

for i, event in enumerate(engine.audit_log, 1):
    print(f"  [{i}] {event.operation:16s} → {event.policy_state:5s}  "
          f"reason={event.reason:24s}  terminal={event.terminal_state}")

# Invariant check: denied/paused events must never be 'executed'
violations = [e for e in engine.audit_log
              if e.policy_state in ('deny', 'pause') and e.terminal_state == 'executed']
print(f"\nExecution invariant violations: {len(violations)}")
assert len(violations) == 0, "FAIL: denied/paused action reached execution!"

## Part 5 — Adversarial Injection

Now let's deliberately attack the policy with forged, expired, and
mismatched approvals, plus malformed arguments and budget exhaustion.

In [ ]:
store = ApprovalStore()
engine = PolicyEngine(budget=20, approval_store=store)

# Scenario 1: Permitted read
d1 = engine.evaluate(acme_employee, read_receipt, now=NOW)
print(f"1. Read receipt:  {d1.state.value:5s}  reason={d1.reason}")

# Scenario 2: Unknown subject
d2 = engine.evaluate(unknown_actor, read_receipt, now=NOW)
print(f"2. Unknown actor: {d2.state.value:5s}  reason={d2.reason}")

# Scenario 3: Cross-tenant resource
cross_tenant = ActionProposal("read_receipt", "receipt-200")
d3 = engine.evaluate(acme_employee, cross_tenant, now=NOW)
print(f"3. Cross-tenant:  {d3.state.value:5s}  reason={d3.reason}")

# Scenario 4: Submit without approval → pause
d4 = engine.evaluate(acme_employee, submit_claim, now=NOW)
print(f"4. No approval:   {d4.state.value:5s}  reason={d4.reason}")

# Scenario 5: Submit with valid approval → allow
valid_receipt = store.issue(
    acme_employee, submit_claim, now=NOW, minutes_valid=30,
)
d5 = engine.evaluate(acme_employee, submit_claim, valid_receipt, now=NOW)
print(f"5. With approval: {d5.state.value:5s}  reason={d5.reason}")

### Nested Resource Bypass (Confused Deputy)

A permitted primary resource (`proposal.resource_id`) is not enough to authorize the operation if it acts on other nested resources. Type validation is insufficient because an attacker could provide structurally correct IDs that belong to a different tenant.

Let us observe the policy denying an authorized primary resource that hides a cross-tenant reference inside `receipt_ids`:

In [ ]:
# Scenario 6: Confused deputy with nested cross-tenant resource
confused_deputy = ActionProposal(
    operation="calculate_total",
    resource_id="receipt-101",   # Acme resource (safe)
    arguments={
        "receipt_ids": [
            "receipt-101",       # Acme
            "receipt-200",       # Globex (Cross-tenant!)
        ]
    }
)

d6 = engine.evaluate(acme_employee, confused_deputy, now=NOW)
print(f"6. Confused deputy: {d6.state.value:5s}  reason={d6.reason}")

## Part 6 — Budget Exhaustion

The per-run budget limits blast radius.  Let's exhaust it.

In [ ]:
budget_engine = PolicyEngine(budget=2, approval_store=store)

d_first = budget_engine.evaluate(
    acme_employee, ActionProposal("read_receipt", "receipt-101"), now=NOW,
)
print(f"First read:  {d_first.state.value}  (budget left: {budget_engine.budget_remaining})")

d_second = budget_engine.evaluate(
    acme_employee, ActionProposal("read_receipt", "receipt-102"), now=NOW,
)
print(f"Second read: {d_second.state.value}  (budget left: {budget_engine.budget_remaining})")

d_third = budget_engine.evaluate(
    acme_employee, ActionProposal("read_receipt", "receipt-101"), now=NOW,
)
print(f"Third read:  {d_third.state.value}  reason={d_third.reason}")

assert d_third.state == Decision.DENY
assert d_third.reason == "budget_exhausted"
print("\n✅ Budget enforcement works.")

## Part 7 — Full Evaluation Table

Let's run the complete built-in evaluation from the lab module.

In [ ]:
lab.run_demo()

## Part 8 — Real SDK Boundary

Now inspect a real OpenAI Agents SDK function tool backed by a strict Pydantic model. This step registers the tool but does **not** call a model or API. The SDK shapes input and can pause orchestration; the application policy still decides authority.

In [ ]:
sdk_lab = importlib.import_module("01_tool_policy_sdk")
sdk_schema = sdk_lab.tool_schema()

print(f"Registered tool: {sdk_lab.submit_expense_claim.name}")
print(f"Approval interrupt enabled: {sdk_lab.submit_expense_claim.needs_approval}")
print(f"Required fields: {sdk_schema['required']}")
print(f"Extra fields accepted: {sdk_schema.get('additionalProperties')}")

assert sdk_lab.submit_expense_claim.needs_approval is True
assert sdk_schema['additionalProperties'] is False
assert set(sdk_schema['required']) == {'claim_id', 'amount'}

### Approval Must Bind the Exact Proposal

A human approved `250.0`. If an edit, race, or compromised component changes it to `251.0`, the proposal digest changes. The policy must deny the changed call and leave the receipt available for the exact proposal.

In [ ]:
import json

sdk_actor = make_actor("emp-42", run_id="notebook-sdk-run")
approved_proposal = ActionProposal("submit_claim", "claim-501", {"amount": 250.0})
sdk_store = ApprovalStore()
sdk_approval = sdk_store.issue(sdk_actor, approved_proposal, now=NOW)
sdk_runtime = sdk_lab.ExpenseRuntime(
    actor=sdk_actor,
    engine=PolicyEngine(approval_store=sdk_store),
    approval=sdk_approval,
    now=NOW,
)

altered = sdk_lab.dispatch_json(
    sdk_runtime, json.dumps({"claim_id": "claim-501", "amount": 251.0})
)
exact = sdk_lab.dispatch_json(
    sdk_runtime, json.dumps({"claim_id": "claim-501", "amount": 250.0})
)

print(f"Altered proposal: {altered.decision} / {altered.reason}")
print(f"Exact proposal:   {exact.decision} / {exact.reason} / executed={exact.executed}")
assert altered.reason == "approval_proposal_mismatch"
assert exact.decision == "allow" and exact.executed

## Part 9 — Metrics With Explicit Populations

Security rates are meaningful only when their populations are visible. The cases below reuse Part 5 and report the numerator and denominator for every metric. These are deterministic fixture results—not a model or production benchmark.

In [ ]:
expected_states = ['allow', 'deny', 'deny', 'pause', 'allow', 'deny']
decisions = [d1, d2, d3, d4, d5, d6]
observations = [
    lab.EvaluationObservation(
        expected_state=expected,
        actual_state=decision.state.value,
        terminal_state=event.terminal_state,
        correlation_id=event.correlation_id,
        reason=event.reason,
        proposal_digest=event.proposal_digest,
        policy_version=event.policy_version,
    )
    for expected, decision, event in zip(expected_states, decisions, engine.audit_log)
]
metrics = calculate_evaluation_metrics(observations)

print(f"Decision accuracy:           {metrics.correct_decision_count}/{metrics.case_count}")
print(f"Unauthorized execution rate: {metrics.unauthorized_execution_count}/{metrics.attack_case_count}")
print(f"Valid-task success rate:     {metrics.valid_task_success_count}/{metrics.valid_case_count}")
print(f"Trace coverage:              {metrics.trace_complete_count}/{metrics.case_count}")
assert metrics.unauthorized_execution_count == 0
assert metrics.valid_task_success_count == metrics.valid_case_count

## Exercises

### Exercise 1 — Add, Do Not Broaden

Add a `cancel_claim` operation with a dedicated scope, resource transition rule, high risk, cost, exact approval, and tests. Keep the existing `submit_claim` contract unchanged.

### Exercise 2 — Dynamic Availability Is Not Authorization

Add a low-risk read tool and expose it only to actors with its scope. Then keep the same scope and resource checks at invocation time. Explain why tool discovery may be stale and cannot grant authority.

### Exercise 3 — External Policy Adapter

Define a policy-provider interface and implement a small OPA, Cedar, or Casbin adapter while preserving the existing decision, reason, policy-version, and audit contract. Fail closed when the provider is unavailable.

### Exercise 4 — Approval and Recovery

Test an edited amount, expired receipt, replay, two concurrent consumers, and a policy-version change. For an executor timeout, define how an idempotency key and effect reconciliation prevent a duplicate reimbursement.

## Production Upgrade Table

| Teaching Shortcut | Production Requirement |
|---|---|
| In-memory subject registry | Real IAM / identity provider |
| In-memory resource registry | Database or catalog lookup |
| Locked in-memory, exactly bound approval | Durable transaction or signed/server-stored one-use receipt |
| In-memory budget counter | Atomic, durable, distributed budget |
| Deterministic side-effect stub | Idempotent tool call plus effect reconciliation |
| One-process synchronization | Distributed concurrency control and durable workflow state |
| No persistent audit | Tamper-resistant, append-only audit storage |

## Next Steps

Continue to the next beginner module — **Prompt Injection** — where you
will learn to recognise and contain untrusted instructions in documents,
tool outputs, and memory.